# M2.2_热传导时间推进与稳定性实验 — 交互式实验工作台（由 HTML 演示网页生成）

> **所属模块**：`02_M2_手写离散_FDM-FVM`  
> **实验定位**：工科本科生计算物理与 SciML 交互式上机实验  
> **运行环境**：本地 Jupyter / VS Code / Colab / 个人笔记本 CPU  

---

## 本讲演示页序导览（Notebook 章节与之严格一一对应）

1. M2.2 热传导时间推进与稳定性实验
2. ① 工程场景与物理直觉：厚壁件的瞬态加热曲线
3. ② 失效先行： $r=0.6$ 的最高频爆炸
4. ③ 三种时间格式的统一写法
5. ④ 显式实现与失效扫描
6. ⑤ 隐式与 CN：稳定性不替代精度
7. ⑥ 验证三件套：与 M1.3 精确解对照
8. ⑦ POE 预测表
9. ⑧ 现场诊断挑战：稳定不等于物理正确
10. ⑨ 验收清单
11. 总结与下一站

> 对齐说明：本 Notebook 由 `slides/*_演示网页.html` 自动生成；
> 代码一律从 `code/solution.py` 真实导入，函数签名由磁盘源码决定，不会漂移。


## 逐页理论与实验（与演示页序一致）

### 第 2 页 · ① 工程场景与物理直觉：厚壁件的瞬态加热曲线

- 热处理炉里的厚壁件、锂电池快充时的电芯温升、建筑墙体的昼夜传热，工程师要的都不是"最终稳态温度"，而是 温度随时间怎么走 ：多久升到目标值的 90%、会不会出现局部过热、停热之后降温有多快。
- 这类问题必须做时间推进，于是立刻引出三个会互相打架的要求—— 算得稳 （不爆炸）、 算得准 （收敛阶对）、 算得平滑 （不出现非物理振荡）。本节的要点是：这三条不是同一件事，满足其中一条不代表满足另一条。
- 30 秒思考 ：工艺要求时间分辨率提高一倍，你会把  $\Delta t$  减半；但若同时为看清边界层又把  $\Delta x$  减半，扩散数  $r=\alpha\Delta t/\Delta x^2$  会变成原来的几倍？


### 第 3 页 · ② 失效先行： $r=0.6$ 的最高频爆炸

- 初值即使是光滑正弦，浮点数仍带有约  $10^{-16}$  的高频舍入误差。显式格式对 Fourier 模态的放大因子为
- $G(\theta)=1-4r\sin^2(\theta/2).$
- 当  $r>1/2$  时，最高频模态的  $|G|>1$ ，误差每步放大；因此“前几十步还平滑”并不等于稳定。
- F2 失效卡 ：记录首次越界步数、 $\max|u|$ 、热量和能量曲线。不要只截取尚未爆炸的时间窗口。
- 完整现场与实测表见  diagnostics/M2.2_F2_F4_cards.md ；全部数字可用  python tests/test_heat_time_stepping.py --dump  在本机复现。
- 30 秒思考 ：网格从 101 点加密到 201 点，如果时间步不改，扩散数  $r$  会如何变化？显式方案会更准确，还是可能从稳定变成失稳？


### 第 4 页 · ③ 三种时间格式的统一写法

- 空间离散写成  $\dot u=L_hu$ ，则：
- 显式 Euler ： $u^{n+1}=(I+\Delta tL_h)u^n$ ；
- 后向 Euler ： $(I-\Delta tL_h)u^{n+1}=u^n$ ；
- Crank–Nicolson ： $(I-\tfrac12\Delta tL_h)u^{n+1}=(I+\tfrac12\Delta tL_h)u^n$ 。
- 后两者每步需要求解线性系统。 A-稳定 指无论时间步多大、数值解都不被放大（后向 Euler 与 CN 都是 A-稳定）；但 A-稳定只保证不爆炸，不保证精度，也不消除方波初值下的高频振铃。


### 第 5 页 · ④ 显式实现与失效扫描

- python · 显式实现与失效扫描
- 可编辑 · Shift+Enter 运行  ↺ 重置  ▶ 运行
- import  numpy  as  np
- from  pathlib  import  Path
- import  importlib.util
- # 定位本单元：单元根目录或项目根目录启动 Jupyter 都能找到 code/solution.py


### 第 6 页 · ⑤ 隐式与 CN：稳定性不替代精度

- python · 隐式与 CN：稳定性不替代精度
- 可编辑 · Shift+Enter 运行  ↺ 重置  ▶ 运行
- from  pathlib  import  Path
- import  importlib.util
- # 定位本单元：单元根目录或项目根目录启动 Jupyter 都能找到 code/solution.py
- _cands = [p /  "code"   for  p  in  (Path.cwd(), *Path.cwd().parents)]


### 第 7 页 · ⑥ 验证三件套：与 M1.3 精确解对照

- 对  $u(x,0)=\sin(\pi x)$ 、零 Dirichlet 边界，解析解是
- $u(x,t)=e^{-\alpha\pi^2t}\sin(\pi x).$
- 验证流程：固定终止时间，分别改变  $\Delta t$  和  $\Delta x$ ，计算  $L^2$  误差；再检查最大值是否超过初值包络，以及总热量是否符合边界条件。
- 方波初值用于暴露 CN 高频振铃；正弦初值用于测量清晰的时间/空间收敛。两个基准不能混用成一张“万能误差图”。


### 第 8 页 · ⑦ POE 预测表

- 💡  POE 自检台 ：先独立思考（可点击填写预测），再点击右侧按钮揭晓标准解析对照。
- 👁️ 全部揭晓 / 隐藏
- 运行前预测
- 观测证据
- 解释/失效归类
- 自检操作


### 第 9 页 · ⑧ 现场诊断挑战：稳定不等于物理正确

- 现场背景： 故障一 · 解是「稳」的，热量方向却反了 ——显式热传导程序在  $r=0.4$  下运行：曲线光滑稳定、无 NaN，但边界热流方向与物理直觉相反——热量从冷端流向热端。 故障二 · 无条件稳定，就可以放心取大时间步吗 ——厚壁件加热曲线交付评审：为提效把 Crank–Nicolson 的扩散数取到  $r=5$ ，解没有爆炸也没有负值，但升温曲线在起始阶段出现明显的非物理「台阶」。请诊断核心病因：
- 课程立场 ：AI 可以把残差和能量曲线画得再漂亮，也不能替你确认边界通量的物理方向。


### 第 10 页 · ⑨ 验收清单

- 能推导 1D/2D 扩散数稳定阈值；
- 能运行交付 API：显式  ftcs_explicit 、后向 Euler  backward_euler 、CN  crank_nicolson  三种推进；
- 能用正弦精确解测量时间/空间收敛；
- 能用方波实验识别 CN 的高频振铃；
- 能把爆炸归入 F2，把符号/单位问题归入 F4；
- 通过 G2（手写数值离散关卡）： python -m pytest tests/ -q  全绿、 --dump  与失效卡逐位一致；


### 第 11 页 · 总结与下一站

- 显式格式的稳定边界来自最高频模态，隐式格式的稳健性也要配合误差和物理检查。下一节  M2.3 波动方程差分、CFL 与色散  将把扩散的无限快平滑换成有限速度传播，并用能量和相位误差区分 CFL 爆炸与数值色散。


## 演示页代码实验台
下面单元格直接抽取自演示网页的可运行代码（`cell-src`），按顺序执行即可复现网页上的交互结果。

In [ ]:

import numpy as np
from pathlib import Path
import importlib.util

# 定位本单元：单元根目录或项目根目录启动 Jupyter 都能找到 code/solution.py
_cands = [p / "code" for p in (Path.cwd(), *Path.cwd().parents)]
_cands += [q / "code" for p in (Path.cwd(), *Path.cwd().parents) for q in p.glob("*/M2.2*")]
unit_root = next(c.parent for c in _cands if (c / "solution.py").is_file())
spec = importlib.util.spec_from_file_location("m22_solution", unit_root / "code" / "solution.py")
S = importlib.util.module_from_spec(spec)
spec.loader.exec_module(S)
nx = 101; x = np.linspace(0, 1, nx); dx = x[1] - x[0]; u0 = np.sin(np.pi*x)
for r in (0.4, 0.5, 0.6):
    dt = r * dx**2
    u = S.ftcs_explicit(u0, 1.0, dx, dt, 250)
    peak = np.max(np.abs(u))
    state = "爆炸" if (not np.all(np.isfinite(u)) or peak > 1e6) else "稳定"
    print(f"r={r:.2f}, max={peak:.3e}, state={state}")
  

In [ ]:

from pathlib import Path
import importlib.util

# 定位本单元：单元根目录或项目根目录启动 Jupyter 都能找到 code/solution.py
_cands = [p / "code" for p in (Path.cwd(), *Path.cwd().parents)]
_cands += [q / "code" for p in (Path.cwd(), *Path.cwd().parents) for q in p.glob("*/M2.2*")]
unit_root = next(c.parent for c in _cands if (c / "solution.py").is_file())
spec = importlib.util.spec_from_file_location("m22_solution", unit_root / "code" / "solution.py")
S = importlib.util.module_from_spec(spec)
spec.loader.exec_module(S)

alpha = 1.0
nx = 101
x = np.linspace(0.0, 1.0, nx)
dx = x[1] - x[0]
u0 = np.sin(np.pi * x)
r = 5.0
dt = r * dx**2 / alpha
n_steps = 40
u_be = S.backward_euler(u0, alpha, dx, dt, n_steps)
u_cn = S.crank_nicolson(u0, alpha, dx, dt, n_steps)
T = n_steps * dt
u_ref = S.exact_heat(x, T, a=np.sqrt(alpha))
for name, u in (("Backward Euler", u_be), ("Crank-Nicolson", u_cn)):
    rel_l2 = np.linalg.norm(u - u_ref) / np.linalg.norm(u_ref)
    print(f"{name}: max={np.max(np.abs(u)):.3e}, relL2={rel_l2:.3e}")
  

## 加载本单元真实源码并核对函数签名
下面直接导入 `code/solution.py`（**不是副本**）；若你在磁盘上改了代码，这里的签名会立刻跟着变。

In [ ]:
# --- 先定位本单元目录：从 notebooks/ 或项目根目录启动 Jupyter 都能找到 code/solution.py
from pathlib import Path
import os
import sys

search_roots = (Path.cwd(), *Path.cwd().parents)
code_candidates = [parent / "code" for parent in search_roots]
for root in search_roots:
    code_candidates.extend(root.glob("*/*/code"))
code_candidates = list(dict.fromkeys(p.resolve() for p in code_candidates))
found = [p.resolve() for p in code_candidates if (p / "solution.py").is_file()]
# 从项目根目录启动时会有多个单元命中，优先锁定本单元（M2.2），避免误载他单元的 solution.py
preferred = [p for p in found if any("M2.2" in part for part in p.parts)]
CODE_DIR = (preferred or found or [None])[0]
if CODE_DIR is None:
    raise FileNotFoundError("找不到 code/solution.py，请从本单元目录或项目根目录启动 Jupyter")
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
ASSET_DIR = CODE_DIR.parent / "assets"
print("CODE_DIR  =", CODE_DIR)
print("ASSET_DIR =", ASSET_DIR)


In [ ]:
import sys, inspect, importlib.util, os
SOL_PATH = os.path.join(CODE_DIR, "solution.py")
MODULE_NAME = "course_unit_M2_2_solution"
spec = importlib.util.spec_from_file_location(MODULE_NAME, SOL_PATH)
S = importlib.util.module_from_spec(spec)
# Make paths relative to the owning unit so main() is independent of Jupyter's cwd.
_OLD_CWD = os.getcwd()
try:
    os.chdir(CODE_DIR.parent)
    spec.loader.exec_module(S)
finally:
    os.chdir(_OLD_CWD)
sys.modules[MODULE_NAME] = S
print("已从磁盘加载:", SOL_PATH)
print()
def _show():
    import ast
    tree = ast.parse(open(SOL_PATH, encoding="utf-8").read())
    for n in tree.body:
        if isinstance(n, ast.FunctionDef) and not n.name.startswith("_"):
            print("def %s" % n.name, inspect.signature(getattr(S, n.name)))
        elif isinstance(n, ast.ClassDef):
            print("class", n.name)
            for m in n.body:
                if isinstance(m, ast.FunctionDef) and (not m.name.startswith("_") or m.name == "__init__"):
                    print("    ", m.name, [a.arg for a in m.args.args])
_show()

## 运行主实验
执行下方单元格即调用本单元真实入口 `main()`，输出与演示网页一致。

In [ ]:
# 执行本单元真实入口函数（网页上的数字由此产生）
try:
    S.main()
except Exception as e:
    print("运行未成功，请把完整报错发给老师：")
    import traceback; traceback.print_exc()

## 物理场成果图展示（相对路径，换机可用）

In [ ]:
from IPython.display import Image, display
import glob
pngs = sorted(glob.glob(os.path.join(ASSET_DIR, "*.png")))
if not pngs:
    print("assets 目录下暂无 png；请先运行上面的求解入口生成图表。")
for f in pngs:
    print("展示:", os.path.basename(f))
    display(Image(filename=f))

## 4. 工科思考与参数敏感度探索 (Exploration Challenges)

### 诊断与思考题：
1. **亲历扩散数越界的数值爆炸现场**：在显式时间推进代码中，固定空间步长 $\Delta x = 0.05$，将时间步长从稳定条件 $r = \alpha \Delta t / \Delta x^2 = 0.4$ 增大至 $r = 0.6$（越界 20%）。重新运行代码，**务必推进到 300 步以上**：前 15 步曲线光滑得看不出任何问题（失效卡的「陷阱段」），约第 130 步首次越界（$\max|u|>10$），300 步后振幅飙到 $10^{26}$ 量级 —— 增长从 $10^{-16}$ 舍入噪声起步，步数给不足就会漏判（这正是本单元的第一条纪律）。
2. **隐式格式无条件稳定的工程代价**：采用 Backward Euler 或 Crank-Nicolson 隐式格式推进时间步时，即使放大时间步长使得 $r = 5.0$，计算也不会爆炸。既然隐式格式永远稳定，为什么在航空发动机燃烧室等瞬态仿真中，时间步长仍然不能设置过大？（时间一阶/二阶截断误差导致物理瞬态峰值被严重低估平滑）。
3. **三对角追赶法 (Thomas Algorithm) 效率神话**：隐式格式每个时间步都需要求解一个大型三对角线性方程组。为什么利用追赶法可以将标准高斯消元的 $\mathcal{O}(N^3)$ 复杂度直接降低到极致的 $\mathcal{O}(N)$ 线性复杂度？